# Zeitungs-Artikel-Extraktion aus gescannten PDFs

Dieses Notebook extrahiert einzelne Artikel aus gescannten Zeitungs-PDFs
(getestet an: *Shanghai Jewish Chronicle*, Internet-Archive-Scan) und
bereitet sie für eine spätere Sentiment-Analyse mit dem NRC Emotion
Lexicon vor.

**Pipeline-Überblick:**

1. PDF-Seite → hochauflösendes Bild rendern (PyMuPDF)
2. Eigenes OCR mit Tesseract (Deutsch **und** Englisch kombiniert)
3. Wörter → Zeilen → Artikel gruppieren (Schriftgröße als Headline-Signal)
4. Anzeigen/Werbung herausfiltern (Chiffre-Nummern, Telefonnummern, ...)
5. Datum/Jahr aus dem Zeitungskopf extrahieren
6. Sprache pro Artikel erkennen (Deutsch vs. Englisch — die Zeitung
   erscheint laut eigenem Masthead als *"English and German Editions"*)
7. Hilfsfunktionen fürs spätere NRC-Lexikon (Umlaut-Ersatzschreibung wie
   `fuer` statt `für`, historisch im Bleisatz üblich)
8. Hauptpipeline: eine Seite / ein PDF / ein ganzer Ordner mit PDFs,
   wahlweise parallelisiert über mehrere CPU-Kerne
9. Stichprobenvalidierung

**Wichtiger Hinweis:** Die Schwellwerte (Headline-Erkennung, Anzeigen-Filter)
sind an einer Testseite kalibriert. Vor der Interpretation echter Ergebnisse
unbedingt Abschnitt 9 (Stichprobenprüfung) nutzen — idealerweise getrennt
für verschiedene Jahrgänge, da sich das Zeitungslayout über die Jahre
verändert hat (siehe Kommentare weiter unten).


## 0. Setup

Benötigte Pakete:
- `pymupdf` — PDF-Seiten als Bilder rendern
- `pandas` — Datenhaltung/CSV
- `langdetect` — Spracherkennung pro Artikel
- das **Tesseract-CLI** (`tesseract`) mit installierten Sprachmodellen
  `deu.traineddata` und `eng.traineddata`

Falls die Sprachmodelle noch fehlen, siehe die auskommentierten
Download-Befehle unten.


In [ ]:
# Falls noch nicht installiert:
# !pip install pymupdf pandas langdetect --quiet

# Falls die Tesseract-Sprachmodelle fehlen (Beispiel für Linux):
# !curl -sL -o deu.traineddata https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/main/deu.traineddata
# !curl -sL -o eng.traineddata https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/main/eng.traineddata
# Danach beide Dateien in das Tesseract-tessdata-Verzeichnis kopieren
# (z.B. /usr/share/tesseract-ocr/5/tessdata/ -- Pfad mit `tesseract --list-langs` bzw.
# der Fehlermeldung von Tesseract pruefen).


In [ ]:
import logging
import os
import re
import subprocess
import tempfile
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, TimeoutError

import fitz  # PyMuPDF
import pandas as pd
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # macht langdetect deterministisch (sonst zufaellig)

# Logging: schreibt Fortschritt sowohl in eine Datei (extraction.log) als
# auch in die Notebook-Ausgabe -- wichtig bei laengeren Laeufen, damit man
# hinterher nachvollziehen kann, welche Seite wie lange gedauert hat bzw.
# wo ein Fehler auftrat.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("extraction.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,  # ueberschreibt evtl. schon vorhandene Logging-Konfiguration
)
log = logging.getLogger(__name__)


## 1. Konfiguration

Alle Schwellwerte und Muster an einer Stelle, damit sie sich leicht
anpassen lassen (z. B. wenn du feststellst, dass ein späterer Jahrgang
dichter gesetzt ist und andere Werte braucht).


In [ ]:
# --- OCR-Einstellungen -------------------------------------------------
DPI = 200                    # Kompromiss aus OCR-Qualitaet und Laufzeit.
                              # Hoehere DPI bringt kaum noch Qualitaetsgewinn,
                              # aber deutlich mehr Rechenzeit.
LANG = "deu+eng"              # Kombiniertes Sprachmodell: die Zeitung erscheint
                              # laut eigenem Masthead als "English and German
                              # Editions" -- ca. 11% der Artikel im Testkorpus
                              # sind tatsaechlich englisch.
MIN_WORD_CONF = 30           # Tesseract-Konfidenz-Schwelle (0-100) pro Wort

# --- Artikel-Segmentierung ----------------------------------------------
HEADLINE_FACTOR = 1.6        # Eine Zeile gilt als Headline (= Artikelanfang),
                              # wenn ihre Hoehe > Median-Zeilenhoehe * Faktor ist.
MIN_ARTICLE_WORDS = 40       # Kuerzere Textbloecke werden verworfen
                              # (isolierte Ueberschriften, Bildunterschriften, ...).

# OCR-Muell-Filter auf Wortebene: einzelne Satzzeichen, extrem lange
# "Woerter" (verklebte Zeilen ohne Leerzeichen) etc. Diese verzerren sowohl
# die Zeilenhoehen-Statistik (Headline-Erkennung) als auch spaeter das
# NRC-Scoring, deshalb schon hier raus.
MIN_WORD_LEN = 1
MAX_WORD_LEN = 40

# --- Anzeigen-Erkennung --------------------------------------------------
# "Chiffre" (anonyme Kontakt-Chiffrenummer) ist in diesem Korpus praktisch
# ausschliesslich in Kleinanzeigen zu finden -- ein sehr zuverlaessiges Signal.
AD_PATTERNS = [
    r"\bChiffre\b",
    r"\bTel\.?\s?\d{4,6}\b",
    r"\bPhone\s?\d{4,6}\b",
    r"\b(Road|Rd\.?|Strasse|Str\.?)\s+\d{1,4}\b",
    r"\bPreis(e)?\s?\$",
    r"\bzu\s+verkaufen\b",
    r"\bgesucht\b.{0,15}\bChiffre\b",
]
AD_REGEX = re.compile("|".join(AD_PATTERNS), re.IGNORECASE)

# Agentur-Kuerzel, die typischerweise direkt unter einer Artikel-Headline
# stehen -- kein Ad-Filter, sondern ein Qualitaetsmerkmal ("das ist mit
# hoher Wahrscheinlichkeit ein echter Nachrichtenartikel").
AGENCY_MARKERS = [
    "Reuter", "Transocean", "Havas", "D.N.B.", "DNB", "CPS", "CPS-Berlin",
    "Central Press",
]

# --- Datumserkennung -----------------------------------------------------
# Regex fuer das Erscheinungsdatum im Zeitungskopf, z.B.
# "Shanghai Donnerstag 22. Juni 1939". Toleriert haeufige OCR-Fehler wie
# "i939" statt "1939" (die fuehrende "1" wird oft als "i"/"l" gelesen).
WEEKDAYS = r"(Montag|Dienstag|Mittwoch|Donnerstag|Freitag|Samstag|Sonnabend|Sonntag)"
MONTHS = r"(Januar|Februar|Maerz|M[\u00e4a]rz|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)"
DATE_PATTERN = re.compile(
    rf"{WEEKDAYS}\.?,?\s+(\d{{1,2}})\.?\s*{MONTHS}\.?,?\s*[il]?(\d{{3,4}})",
    re.IGNORECASE,
)
MONTH_NUM = {
    "januar": 1, "februar": 2, "maerz": 3, "m\u00e4rz": 3, "april": 4, "mai": 5,
    "juni": 6, "juli": 7, "august": 8, "september": 9, "oktober": 10,
    "november": 11, "dezember": 12,
}

print("Konfiguration geladen.")


## 2. Schritt 1+2: PDF-Seite rendern und OCRen

Statt die im PDF eingebettete OCR-Textebene zu nutzen (in Tests: durcheinander-
gewürfelte Lesereihenfolge, viele Fehlzeichen), rendern wir jede Seite als
hochauflösendes Bild und lassen Tesseract selbst darüber laufen.

Der Trick: Tesseract liefert über die **TSV-Ausgabe** nicht nur den reinen
Text, sondern auch **Layout-Informationen** pro Wort — `block_num`/`par_num`/
`line_num` (Tesseracts eigene Spalten-/Absatz-Erkennung) sowie die
Bounding-Box und eine Konfidenz. Genau diese Information brauchen wir
gleich, um Zeilenhöhen (= Schriftgröße) zu bestimmen und damit Headlines
zu erkennen.


In [ ]:
def render_page(doc, page_num, dpi=DPI, out_path=None):
    """Rendert eine PDF-Seite als PNG."""
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    out_path = out_path or tempfile.mktemp(suffix=".png")
    pix.save(out_path)
    return out_path


def ocr_page_tsv(image_path, lang=LANG):
    """Ruft Tesseract auf und gibt die Wort-Ebene als DataFrame zurueck.

    Die TSV-Ausgabe enthaelt pro Wort: block_num/par_num/line_num, die
    Bounding Box (left/top/width/height) und eine Konfidenz (conf, 0-100).
    """
    out_base = tempfile.mktemp()
    try:
        subprocess.run(
            ["tesseract", image_path, out_base, "-l", lang, "--psm", "3", "tsv"],
            check=True, capture_output=True, timeout=120,
        )
        df = pd.read_csv(out_base + ".tsv", sep="\t", quoting=3)
    finally:
        tsv_path = out_base + ".tsv"
        if os.path.exists(tsv_path):
            os.remove(tsv_path)

    df = df[df["level"] == 5]  # nur Wort-Ebene (Level 5 = einzelnes Wort)
    df = df[df["conf"].astype(float) >= MIN_WORD_CONF]
    df = df[df["text"].notna()]
    df = df[df["text"].astype(str).str.len().between(MIN_WORD_LEN, MAX_WORD_LEN)]
    return df

print("OCR-Funktionen definiert.")


## 3. Schritt 3: Wörter → Zeilen → Artikel gruppieren

Drei Stufen:
1. Einzelne OCR-Wörter werden zu **Zeilen** zusammengefasst (gruppiert nach
   Tesseracts `block_num`/`par_num`/`line_num`).
2. Zeilen werden zu **Artikeln** gruppiert: Eine auffällig große Zeile
   (Headline) startet einen neuen Artikel.
3. Kleine Textbereinigung (Mehrfach-Leerzeichen, Bindestrich-Trennungen).


In [ ]:
def words_to_lines(word_df):
    """Gruppiert einzelne OCR-Woerter zu Zeilen."""
    if word_df.empty:
        return pd.DataFrame(columns=["block_num", "par_num", "line_num",
                                      "text", "height", "top", "left"])
    lines = (
        word_df.sort_values(["block_num", "par_num", "line_num", "word_num"])
        .groupby(["block_num", "par_num", "line_num"])
        .agg(
            text=("text", lambda x: " ".join(str(w) for w in x)),
            height=("height", "mean"),
            top=("top", "min"),
            left=("left", "min"),
        )
        .reset_index()
        .sort_values(["block_num", "par_num", "top"])
    )
    return lines


def group_into_articles(lines, headline_factor=HEADLINE_FACTOR,
                         min_words=MIN_ARTICLE_WORDS):
    """Gruppiert Zeilen zu Artikeln. Eine neue, auffaellig grosse Zeile
    (Headline) startet einen neuen Artikel."""
    if lines.empty:
        return []

    median_height = lines["height"].median()
    headline_threshold = median_height * headline_factor

    articles = []
    current_lines = []

    for _, row in lines.iterrows():
        is_headline = row["height"] >= headline_threshold and len(row["text"].split()) <= 12
        if is_headline and current_lines:
            articles.append(_finalize_article(current_lines))
            current_lines = [row]
        else:
            current_lines.append(row)
    if current_lines:
        articles.append(_finalize_article(current_lines))

    # zu kurze "Artikel" (z.B. isolierte Headlines ohne Fliesstext) raus
    articles = [a for a in articles if len(a["text"].split()) >= min_words]
    return articles


def _finalize_article(rows):
    text = " ".join(r["text"] for r in rows) if isinstance(rows, list) else \
        " ".join(rows["text"])
    top = min(r["top"] for r in rows) if isinstance(rows, list) else rows["top"].min()
    return {"text": text.strip(), "top": top}


def clean_text(text):
    """Kleine Textbereinigung: Mehrfach-Leerzeichen entfernen und simple
    Trennstrich-am-Zeilenende-Faelle zusammenfuehren (z.B. "Kran- kheit"
    -> "Krankheit"). Heuristik, nicht immer korrekt, verbessert aber die
    spaetere NRC-Lexikon-Trefferquote spuerbar."""
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("Segmentierungs-Funktionen definiert.")


## 4. Schritt 4: Anzeigen/Werbung herausfiltern

Kombination aus mehreren Signalen: zu kurzer Text, typische Anzeigen-Muster
(„Chiffre", Telefonnummern, Adressmuster), hoher Ziffernanteil.


In [ ]:
def is_advertisement(text, min_words=MIN_ARTICLE_WORDS):
    word_count = len(text.split())
    if word_count < min_words:
        return True
    if AD_REGEX.search(text):
        return True
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    if digit_ratio > 0.12:
        return True
    return False


def has_agency_marker(text):
    return any(marker.lower() in text.lower() for marker in AGENCY_MARKERS)

print("Anzeigen-Filter definiert.")


## 5. Schritt 5: Datum/Jahr aus dem Zeitungskopf extrahieren

Das Datum steht im Masthead jeder Ausgabe (z. B. „Shanghai Donnerstag 22.
Juni 1939"). Die Regex toleriert typische OCR-Fehler bei der Jahreszahl —
die führende „1" wird oft als „i"/"l" gelesen und geht dabei verloren
(„1939" → „i939" → Ziffern-Gruppe nur „939"). Das wird hier erkannt und
auf 4-stellig korrigiert.


In [ ]:
def extract_date(text):
    m = DATE_PATTERN.search(text)
    if not m:
        return None, None
    _, day, month, year = m.groups()
    month_num = MONTH_NUM.get(month.lower().replace("\u00e4", "ae"))
    year = int(year)
    # OCR verschluckt die fuehrende "1" von Jahreszahlen manchmal
    # ("i939" -> Zifferngruppe nur "939"). Auf 4-stellig auffuellen.
    if 100 <= year < 1000:
        year += 1000
    elif year < 100:
        year += 1900
    return year, month_num


# Kurzer Test:
print(extract_date("Shanghai Donnerstag 22 Juni. i939"))  # sollte (1939, 6) ergeben


## 6. Schritt 6: Sprache pro Artikel + NRC-Lexikon-Hilfsfunktionen

Zwei Dinge, die für die spätere Sentiment-Analyse wichtig sind:

**a) Sprache pro Artikel.** Die Zeitung erscheint laut Masthead als
*„English and German Editions"* — im Testkorpus waren rund 11% der
Artikel tatsächlich englisch. Fürs NRC-Scoring muss pro Artikel das
passende Lexikon (Deutsch oder Englisch) verwendet werden, sonst matchen
englische Wörter gegen das deutsche Lexikon (oder umgekehrt) und liefern
bedeutungslose Scores nahe Null.

**b) Umlaut-Ersatzschreibung.** Der historische Zeitungssatz verwendet
durchgängig `ae/oe/ue/ss` statt `ä/ö/ü/ß` (kein Umlaut im damaligen
Bleisatz-/Telegrafie-Zeichensatz verfügbar) — das ist **kein OCR-Fehler**,
sondern so wurde tatsächlich gedruckt. Im Testkorpus kommt allein „fuer"
229 Mal vor. Die Transliteration passiert bewusst **am NRC-Lexikon**
(„über" → „ueber"), nicht umgekehrt am Korpustext: „ue" im Text zu „ü"
zurückzuwandeln wäre mehrdeutig (Wörter wie „Museum" oder „neu" enthalten
„ue"/"eu", ohne dass ein Umlaut gemeint ist).


In [ ]:
def detect_language(text):
    """Erkennt die Sprache eines Artikels (de/en/...). Gibt bei sehr
    kurzen/uneindeutigen Texten "unknown" zurueck statt zu raten."""
    if len(text.split()) < 5:
        return "unknown"
    try:
        return detect(text)
    except Exception:
        return "unknown"


# Ersatzschreibung wie im historischen Satz: kein Umlaut/scharfes S im
# Zeichensatz verfuegbar, daher "ae/oe/ue/ss" statt "ä/ö/ü/ß".
_UMLAUT_MAP = str.maketrans({
    "\u00e4": "ae", "\u00f6": "oe", "\u00fc": "ue",
    "\u00c4": "Ae", "\u00d6": "Oe", "\u00dc": "Ue",
})


def delatinize(word):
    """Transliteriert ein Wort mit 'echten' Umlauten/scharfem S in die im
    Korpus verwendete Ersatzschreibung, z.B. 'ueber' <- 'über',
    'fuer' <- 'für', 'moechte' <- 'möchte', 'Strasse' <- 'Straße'."""
    word = word.replace("\u00df", "ss")
    return word.translate(_UMLAUT_MAP)


def delatinize_lexicon(word2emotions):
    """Erweitert ein geladenes NRC-Lexikon (Wort -> Set von Emotionen) um
    transliterierte Zusatzeintraege, damit 'fuer', 'ueber', 'moechte' im
    Korpus ebenfalls gematcht werden. Bestehende Eintraege bleiben
    erhalten; bei Kollisionen werden die Emotionsmengen vereinigt."""
    extended = dict(word2emotions)
    for word, emotions in word2emotions.items():
        alt = delatinize(word)
        if alt != word:
            extended[alt] = extended.get(alt, set()) | emotions
    return extended


# Kurzer Test:
for w in ["f\u00fcr", "\u00fcber", "k\u00f6nnen", "m\u00f6chte"]:
    print(w, "->", delatinize(w))


## 7. Hauptpipeline: eine Seite / ein PDF / ein ganzer Ordner

Drei Funktionen, aufeinander aufbauend:

- **`process_page()`** — verarbeitet genau eine Seite.
- **`process_pdf()`** — verarbeitet ein ganzes PDF, seriell oder parallel
  über `ProcessPoolExecutor`, mit laufendem Checkpointing **und einem
  harten Timeout pro Seite**.
- **`process_folder()`** — verarbeitet **alle PDFs in einem Ordner** (z. B.
  ein Jahrgang pro Datei), inkl. `source_file`-Spalte.

**Wichtige Lektion aus der Praxis:** Bei einem echten 227-Seiten-Lauf ist
uns eine frühere Version dieses Skripts scheinbar 14 Stunden lang
"hängengeblieben". Ursache vermutlich: (a) **Speicher-Thrashing** — eine
Seite bei 300dpi belegt unkomprimiert ca. 250 MB RAM, bei zu vielen
parallelen Workern reicht der Arbeitsspeicher nicht mehr, das Betriebssystem
fängt an zu swappen, und alles wird um den Faktor 50–100 langsamer statt
schneller; und (b) ein **Design-Fehler**: die ursprüngliche Version nutzte
`pool.starmap()`, das *erst nach allen Seiten* irgendein Ergebnis
zurückgibt — hängt eine einzige Seite, sieht man das nirgends, und der
ganze Lauf steht scheinbar grundlos still.

Diese Version behebt beides:
- **`PAGE_TIMEOUT_SECONDS`**: harte Obergrenze pro Seite (Rendern *und*
  OCR). Hängt eine Seite länger, wird sie übersprungen statt den Lauf zu
  blockieren.
- **Laufendes Checkpointing + Fortschritts-Log** auch im Parallel-Modus:
  alle 5 Seiten eine Log-Zeile mit Stand und vergangener Zeit — kein
  stundenlanges Rätselraten mehr, ob noch etwas passiert.
- **Faustregel für `n_workers`**: `n_workers <= freier RAM in GB / 1.5`.
  Lieber weniger Worker, aber dafür kein Swapping.


In [ ]:
def process_page(pdf_path, page_num, n_pages, source_file=None):
    """Verarbeitet genau eine Seite, gibt eine Liste von Artikel-Dicts
    zurueck."""
    source_file = source_file or os.path.basename(pdf_path)
    page_t0 = time.time()
    articles = []
    try:
        doc = fitz.open(pdf_path)  # pro Prozess neu oeffnen (fitz.Document
                                    # ist nicht sicher ueber Prozessgrenzen
                                    # hinweg teilbar)
        img_path = render_page(doc, page_num)
        try:
            word_df = ocr_page_tsv(img_path)
        finally:
            if os.path.exists(img_path):
                os.remove(img_path)

        lines = words_to_lines(word_df)
        if lines.empty:
            log.info(f"{source_file} Seite {page_num + 1}/{n_pages}: kein Text erkannt.")
            return []

        page_text = " ".join(lines["text"])
        year, month = extract_date(page_text)

        candidates = group_into_articles(lines)
        for art in candidates:
            text = clean_text(art["text"])
            if is_advertisement(text):
                continue
            articles.append({
                "source_file": source_file,
                "pdf_page": page_num + 1,
                "year": year,
                "month": month,
                "lang": detect_language(text),
                "has_agency_marker": has_agency_marker(text),
                "word_count": len(text.split()),
                "text": text,
            })

        log.info(
            f"{source_file} Seite {page_num + 1}/{n_pages}: "
            f"{len(candidates)} Bloecke, {len(articles)} behalten, "
            f"Jahr={year}, {time.time() - page_t0:.1f}s"
        )
    except Exception as exc:
        log.error(f"{source_file} Seite {page_num + 1}/{n_pages}: FEHLER -- {exc!r}")
        return []
    return articles


In [ ]:
PAGE_TIMEOUT_SECONDS = 180  # harte Obergrenze pro Seite (Rendern + OCR
                             # zusammen). Wichtig: das Tesseract-Timeout in
                             # ocr_page_tsv() schuetzt NUR den Tesseract-
                             # Subprozess, nicht das PDF-Rendern
                             # (fitz.get_pixmap()) davor -- haengt oder
                             # thrasht DAS (z.B. bei Speicherdruck durch zu
                             # viele parallele Worker), gab es bisher
                             # keinerlei Schutz und keine Rueckmeldung.
                             # Genau das fuehrt in der Praxis zu Laeufen,
                             # die scheinbar stundenlang "haengen".


def process_pdf(pdf_path, page_range=None, n_workers=1):
    """Verarbeitet ein ganzes PDF und verhindert Pool-Abstürze."""
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    pages_to_process = list(range(n_pages) if page_range is None else page_range)

    all_articles = []
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {
            executor.submit(process_page, pdf_path, p): p for p in pages_to_process
        }
        for future in futures:
            page = futures[future]
            try:
                articles = future.result(timeout=120)  # Timeout für jeden Prozess
                all_articles.extend(articles)
            except TimeoutError:
                logging.error(f"Seite {page + 1} hat Timeout überschritten.")
            except Exception as e:
                logging.error(f"Fehler auf Seite {page + 1}: {e}")
    return all_articles

def process_folder(folder_path, out_path="articles_all.csv", n_workers=1,
                    glob_pattern="*.pdf"):
    """Verarbeitet ALLE PDF-Dateien in einem Ordner und fuehrt die
    Ergebnisse in einer einzigen CSV zusammen."""
    pdf_files = sorted(Path(folder_path).glob(glob_pattern))
    log.info(f"{len(pdf_files)} PDF-Dateien gefunden in {folder_path}")

    all_dfs = []
    for pdf_file in pdf_files:
        checkpoint = f"{pdf_file.stem}_checkpoint.csv"
        log.info(f"=== Starte {pdf_file.name} ===")
        df = process_pdf(str(pdf_file), checkpoint_path=checkpoint, n_workers=n_workers)
        all_dfs.append(df)
        pd.concat(all_dfs, ignore_index=True).to_csv(out_path, index=False)

    final_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
    final_df = final_df.drop_duplicates(subset=["source_file", "pdf_page", "text"])
    final_df.to_csv(out_path, index=False)
    log.info(f"Gesamt: {len(final_df)} Artikel aus {len(pdf_files)} Dateien in {out_path}.")
    return final_df

print("Hauptpipeline definiert.")


## 8. Stichprobenvalidierung

**Wichtig, bevor du die Ergebnisse interpretierst!** Zieht eine
Zufallsstichprobe zur manuellen Prüfung — optional stratifiziert nach Jahr,
weil sich Layout und OCR-Qualität über die Jahre verändern können
(z. B. dichterer Satz und mehr Anzeigen in späteren Kriegsjahren). Eine
rein globale Stichprobe könnte solche zeitraumspezifischen Probleme leicht
verdecken.


In [ ]:
def validate_sample(articles_df, n=15, out_path="sample_for_review.csv",
                     stratify_by=None):
    """Zieht eine Zufallsstichprobe zur manuellen Pruefung. Bei
    stratify_by="year" gleichmaessig ueber die Jahre verteilt."""
    if stratify_by and stratify_by in articles_df.columns:
        groups = articles_df.groupby(stratify_by, dropna=False)
        n_per_group = max(1, n // groups.ngroups)
        sample = groups.apply(lambda g: g.sample(min(n_per_group, len(g)))).reset_index(drop=True)
    else:
        sample = articles_df.sample(min(n, len(articles_df)))
    sample.to_csv(out_path, index=False)
    print(f"{len(sample)} Artikel zur manuellen Pruefung in {out_path} gespeichert.")
    return sample

print("Validierungsfunktion definiert.")


## 9. Ausführung

Drei typische Varianten — je nachdem, was du gerade brauchst. Nur die
jeweils passende Zelle ausführen.

**Vor einem großen Lauf: RAM und Kernzahl checken**, damit `n_workers`
sinnvoll gewählt ist (siehe Warnung in Abschnitt 7):
```bash
free -h    # freier RAM
nproc      # Anzahl CPU-Kerne
```
Faustregel: `n_workers <= freier RAM in GB / 1.5`. Lieber vorsichtig
anfangen (z. B. 2) und bei Bedarf hochsetzen, als von Anfang an zu viele
Worker zu starten und in Swapping/Thrashing zu laufen.

**Falls ein Lauf ungewöhnlich lange dauert**, in einer zweiten Zelle/einem
zweiten Terminal parallel checken:
```bash
tail -20 extraction.log        # kommt noch Fortschritt rein?
ls -la *checkpoint*.csv        # wann zuletzt aktualisiert?
free -h                        # wird geswappt?
```


In [ ]:
import psutil

# Gesamten, verfügbaren und verwendeten Speicher in GB ausgeben
mem_info = psutil.virtual_memory()
total_mem = mem_info.total / (1024 ** 3)  # GB
available_mem = mem_info.available / (1024 ** 3)  # GB
used_mem = mem_info.used / (1024 ** 3)  # GB

print(f"Gesamtspeicher: {total_mem:.2f} GB")
print(f"Verfügbarer Speicher: {available_mem:.2f} GB")
print(f"Genutzter Speicher: {used_mem:.2f} GB")

In [ ]:
# Physische und logische CPU-Kerne abrufen
physical_cores = psutil.cpu_count(logical=False)
logical_cores = psutil.cpu_count(logical=True)

print(f"Physische Kerne: {physical_cores}")
print(f"Logische Kerne: {logical_cores}")

# Maximal mögliche Worker berechnen anhand des verfügbaren RAM
n_workers = int(min(logical_cores, available_mem // 1.5))  # Beispiel: Faustregel von 1.5 GB/Worker
print(f"Empfohlene Anzahl an Workern: {n_workers}")



In [ ]:
# --- Variante A: Testlauf an wenigen Seiten einer einzelnen Datei -------
pdf_path = "shanghaijewishch00unse.pdf"

df = process_pdf(pdf_path, page_range=range(1, 4), n_workers= 1)
df


In [ ]:
articles = process_page("shanghaijewishch00unse.pdf", page_num=3, n_pages=227)
try:
    doc = fitz.open(pdf_path)
    print(f"PDF '{pdf_path}' erfolgreich geöffnet.")
except Exception as e:
    print(f"Fehler beim Öffnen der PDF-Datei: {e}")

try:
    pix = doc[3].get_pixmap(dpi=300)
    pix.save("output.png")
except Exception as e:
    print(f"Fehler beim Rendern der Seite: {e}")



In [ ]:
# --- Variante B: ganzes PDF, parallelisiert -----------------------------
import multiprocessing
print("Verfuegbare CPU-Kerne:", multiprocessing.cpu_count())

df = process_pdf(pdf_path, page_range=None, n_workers=3,
                   checkpoint_path="articles_checkpoint.csv")
df.to_csv("articles_extracted.csv", index=False)


In [ ]:
# --- Variante C: ganzer Ordner mit mehreren PDF-Baenden -----------------
# df = process_folder("/pfad/zu/deinen/pdfs/", out_path="articles_all.csv",
#                      n_workers=4)


In [ ]:
# --- Stichprobe zur manuellen Pruefung ----------------------------------
if len(df) > 0:
    sample = validate_sample(df, n=min(10, len(df)), stratify_by="year")
    sample


## 10. Wie es weitergeht

Die hier erzeugte Artikel-Tabelle (Spalten: `source_file`, `pdf_page`,
`year`, `month`, `lang`, `has_agency_marker`, `word_count`, `text`) ist der
Input für die eigentliche Sentiment-Analyse:

1. NRC-Lexikon laden (Deutsch **und** Englisch, je nach `lang`-Spalte)
2. `delatinize_lexicon()` auf das deutsche Lexikon anwenden
3. Pro Artikel tokenisieren/lemmatisieren (spaCy, passendes Sprachmodell
   je nach `lang`)
4. Emotionsanteile pro Dokument berechnen (normalisiert auf Textlänge)
5. Nach Jahr aggregieren, Artikel je Emotion ranken

Das war Gegenstand der vorherigen Schritte in unserem Gespräch — bei
Bedarf baue ich das gerne als weiteren Notebook-Abschnitt an.